# 🚀 Lab 03 — Orchestration CI/CD avec GitHub Actions & BigQuery

**Objectifs pédagogiques :**
- Comprendre le concept **Serverless** : GitHub provisionne et détruit une VM pour vous
- Créer un **pipeline de données SQL** orchestré par GitHub Actions
- Observer la **séquentialité du DAG** : si Bronze échoue, Silver ne tourne pas
- Automatiser les transformations **Medallion sans aucune dépendance Python** côté CI/CD

**Durée estimée :** 45 min  
**Outils :** Console BigQuery, GitHub (dépôt personnel), `bq` CLI (pré-installé sur les runners GitHub)

> 💡 **L'idée clé de ce lab :**  
> Le CLI `bq` est **pré-installé** sur tous les runners GitHub (`ubuntu-latest`).  
> On n'a donc besoin que de s'authentifier à GCP, puis d'appeler `bq query` avec nos SQL.  
> **Zéro `pip install`, zéro `profiles.yml`, zéro gestion d'environnement.**

---

```
Structure de ce lab :

Partie 0 — Prérequis & préparation du dépôt GitHub
Partie 1 — Écrire les requêtes SQL de transformation
Partie 2 — Créer le workflow GitHub Actions
Partie 3 — Sécuriser les credentials GCP (Secrets)
Partie 4 — Déclencher et observer le pipeline
Partie 5 — Validation & checkpoint
```


---
## ⚙️ Partie 0 — Prérequis & Préparation

### 0.1 — Ce qu'il vous faut avant de commencer

| Prérequis | Statut attendu |
|-----------|---------------|
| Compte GitHub (gratuit) | Connecté sur [github.com](https://github.com) |
| Projet GCP actif | `PROJECT_ID` connu |
| Datasets BigQuery | `bronze`, `silver`, `gold` créés (Lab 02) |
| Table `bronze.raw_events` | Données présentes (Lab 02) |

> ⚠️ Si les datasets n'existent pas, exécutez la cellule suivante depuis Cloud Shell.


In [ ]:
# (Optionnel) Recréer les datasets si nécessaire
export PROJECT_ID=$(gcloud config get-value project)
echo "PROJECT_ID = $PROJECT_ID"

for LAYER in bronze silver gold; do
  bq mk --location=EU --dataset "${PROJECT_ID}:${LAYER}" 2>/dev/null || echo "Dataset ${LAYER} déjà existant"
done
echo "✅ Datasets vérifiés"


### 0.2 — Créer votre dépôt GitHub

1. Allez sur [github.com/new](https://github.com/new)
2. Nommez-le `bigdata-pipeline` (ou autre nom de votre choix)
3. Cochez **"Add a README file"** → créer
4. Clonez-le localement :

```bash
git clone https://github.com/<VOTRE_USERNAME>/bigdata-pipeline.git
cd bigdata-pipeline
```

### 0.3 — Structure de fichiers à créer

Voici l'arborescence finale que vous allez construire dans ce lab :

```
bigdata-pipeline/
├── .github/
│   └── workflows/
│       └── medallion_pipeline.yml   ← Le DAG GitHub Actions
├── sql/
│   ├── 01_bronze_to_silver.sql      ← Transformation Bronze → Silver
│   ├── 02_silver_to_gold.sql        ← Agrégation Silver → Gold
│   └── 03_validate_gold.sql         ← Validation qualité
└── README.md
```

Créez les dossiers maintenant :

```bash
mkdir -p .github/workflows sql
```


---
## 🥈 Partie 1 — Les requêtes SQL de transformation

Chaque étape du pipeline est un **fichier `.sql` indépendant**.  
Le workflow GitHub Actions les exécutera dans l'ordre, via `bq query`.

### 1.1 — Fichier `sql/01_bronze_to_silver.sql`

Copiez ce contenu dans le fichier `sql/01_bronze_to_silver.sql` :


In [ ]:
-- ── sql/01_bronze_to_silver.sql ─────────────────────────────────────────────
-- Transformation Bronze → Silver
-- Logique : typage, déduplication par ROW_NUMBER(), filtrage des invalides
-- Exécuté automatiquement par GitHub Actions à chaque run du pipeline

CREATE OR REPLACE TABLE `silver.events`
PARTITION BY event_date
CLUSTER BY user_id, event_type
OPTIONS (
  description = 'Événements nettoyés — couche Silver. Généré par GitHub Actions.'
)
AS
WITH
-- Étape 1 : typer et normaliser les colonnes brutes
typed AS (
  SELECT
    event_id,
    PARSE_TIMESTAMP('%Y-%m-%dT%H:%M:%SZ', event_ts)   AS event_ts,
    user_id,
    LOWER(TRIM(event_type))                            AS event_type,
    device_os,
    SAFE_CAST(amount_str AS FLOAT64)                   AS amount,
    country,
    _loaded_at
  FROM `bronze.raw_events`
  WHERE
    device_os NOT IN ('UnknownOS')
    AND country != 'XX'
),
-- Étape 2 : dédupliquer (garder l'occurrence la plus récente)
deduped AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY event_id
      ORDER BY event_ts DESC
    ) AS rn
  FROM typed
)
SELECT
  event_id,
  event_ts,
  DATE(event_ts)  AS event_date,
  user_id,
  event_type,
  device_os,
  amount,
  country,
  _loaded_at
FROM deduped
WHERE rn = 1;


### 1.2 — Fichier `sql/02_silver_to_gold.sql`

Copiez ce contenu dans `sql/02_silver_to_gold.sql` :


In [ ]:
-- ── sql/02_silver_to_gold.sql ────────────────────────────────────────────────
-- Agrégation Silver → Gold
-- KPIs quotidiens par pays et OS — prêts pour Looker Studio
-- Exécuté automatiquement par GitHub Actions, APRÈS silver.events

CREATE OR REPLACE TABLE `gold.daily_kpis`
PARTITION BY report_date
CLUSTER BY country
OPTIONS (
  description = 'KPIs quotidiens agrégés — couche Gold. Généré par GitHub Actions.'
)
AS
SELECT
  event_date                                                    AS report_date,
  country,
  device_os,
  COUNT(DISTINCT user_id)                                       AS active_users,
  COUNT(*)                                                      AS total_events,
  COUNT(CASE WHEN event_type = 'purchase_success' THEN 1 END)   AS purchases,
  COUNT(CASE WHEN event_type = 'page_view'        THEN 1 END)   AS page_views,
  COUNT(CASE WHEN event_type = 'add_to_cart'      THEN 1 END)   AS cart_adds,
  ROUND(SUM(amount), 2)                                         AS total_revenue,
  ROUND(
    AVG(CASE WHEN event_type = 'purchase_success' THEN amount END), 2
  )                                                             AS avg_basket,
  CURRENT_TIMESTAMP()                                           AS refreshed_at
FROM `silver.events`
GROUP BY 1, 2, 3
ORDER BY 1 DESC, 2, 3;


### 1.3 — Fichier `sql/03_validate_gold.sql`

Ce fichier est la **porte de qualité** : si une règle échoue, le pipeline s'arrête avec une erreur.  
Copiez ce contenu dans `sql/03_validate_gold.sql` :


In [ ]:
-- ── sql/03_validate_gold.sql ─────────────────────────────────────────────────
-- Validation qualité de la couche Gold
-- Si COUNT(*) = 0 → la requête retourne 0 ligne → le script shell échoue
-- Exécuté automatiquement par GitHub Actions, APRÈS gold.daily_kpis
--
-- Règles vérifiées :
--   1. La table Gold n'est pas vide
--   2. Le revenu total est positif
--   3. Pas de valeurs négatives dans total_revenue

SELECT
  COUNT(*)            AS total_lignes,
  SUM(total_revenue)  AS revenu_total,
  COUNT(CASE WHEN total_revenue < 0 THEN 1 END) AS revenus_negatifs
FROM `gold.daily_kpis`
HAVING
  COUNT(*) > 0                                  -- 1. Table non vide
  AND SUM(total_revenue) > 0                    -- 2. Revenu positif
  AND COUNT(CASE WHEN total_revenue < 0 THEN 1 END) = 0;  -- 3. Pas de négatifs
-- Si HAVING filtre toutes les lignes → 0 résultats → le pipeline échoue ✅


### 1.4 — Tester les requêtes localement (optionnel)

Avant de pousser sur GitHub, vous pouvez valider que les requêtes fonctionnent :


In [ ]:
export PROJECT_ID=$(gcloud config get-value project)

# Tester Bronze → Silver
bq query \
  --project_id="${PROJECT_ID}" \
  --location=EU \
  --use_legacy_sql=false \
  --format=pretty \
  < sql/01_bronze_to_silver.sql

echo "✅ Bronze → Silver : OK"

# Tester Silver → Gold
bq query \
  --project_id="${PROJECT_ID}" \
  --location=EU \
  --use_legacy_sql=false \
  --format=pretty \
  < sql/02_silver_to_gold.sql

echo "✅ Silver → Gold : OK"


---
## 🔐 Partie 2 — Sécuriser les credentials GCP

Pour que GitHub puisse appeler BigQuery, il a besoin d'une **clé de compte de service** GCP.  
Cette clé est stockée comme **Secret chiffré** dans GitHub — jamais dans le code.

### 2.1 — Créer le compte de service


In [ ]:
export PROJECT_ID=$(gcloud config get-value project)
SA_NAME="github-actions-bq"
SA_EMAIL="${SA_NAME}@${PROJECT_ID}.iam.gserviceaccount.com"

# Créer le compte de service
gcloud iam service-accounts create "${SA_NAME}" \
  --display-name="GitHub Actions — BigQuery Pipeline" \
  --project="${PROJECT_ID}"

# Donner uniquement les droits BigQuery nécessaires
gcloud projects add-iam-policy-binding "${PROJECT_ID}" \
  --member="serviceAccount:${SA_EMAIL}" \
  --role="roles/bigquery.dataEditor"

gcloud projects add-iam-policy-binding "${PROJECT_ID}" \
  --member="serviceAccount:${SA_EMAIL}" \
  --role="roles/bigquery.jobUser"

echo "✅ Compte de service créé : ${SA_EMAIL}"
echo "Rôles attribués : bigquery.dataEditor + bigquery.jobUser"


### 2.2 — Générer la clé JSON


In [ ]:
export PROJECT_ID=$(gcloud config get-value project)
SA_EMAIL="github-actions-bq@${PROJECT_ID}.iam.gserviceaccount.com"

# Générer la clé JSON
gcloud iam service-accounts keys create /tmp/gcp_key.json \
  --iam-account="${SA_EMAIL}"

echo "✅ Clé générée dans /tmp/gcp_key.json"
echo ""
echo "📋 Contenu à copier pour le Secret GitHub :"
echo "────────────────────────────────────────────"
cat /tmp/gcp_key.json
echo "────────────────────────────────────────────"
echo "⚠️  NE COMMITEZ JAMAIS ce fichier dans Git !"


### 2.3 — Enregistrer le Secret dans GitHub

**Étapes à faire sur github.com :**

1. Allez sur votre dépôt → onglet **Settings**
2. Menu gauche : **Secrets and variables → Actions**
3. Cliquez **New repository secret**
4. Remplissez :
   - **Name** : `GCP_SA_KEY`
   - **Secret** : Collez le contenu JSON affiché ci-dessus
5. Cliquez **Add secret**

Egalement, créez un second secret :
- **Name** : `GCP_PROJECT_ID`  
- **Secret** : Votre ID de projet GCP (ex: `lasalle-bigdata-2024`)

> 🔐 **Pourquoi des Secrets ?**  
> Les Secrets GitHub sont chiffrés et ne sont **jamais visibles** dans les logs du pipeline.  
> Ils sont injectés comme variables d'environnement uniquement au moment de l'exécution.

```
GitHub Repository
├── Settings → Secrets and variables → Actions
│   ├── GCP_SA_KEY        ← contenu JSON de /tmp/gcp_key.json
│   └── GCP_PROJECT_ID    ← ex: lasalle-bigdata-2024
│
└── .github/workflows/medallion_pipeline.yml
    └── ${{ secrets.GCP_SA_KEY }}   ← jamais en clair dans le code
```


---
## 🔧 Partie 3 — Le Workflow GitHub Actions

C'est le cœur du lab : le fichier YAML qui définit le **DAG d'orchestration**.  
Copiez ce contenu dans `.github/workflows/medallion_pipeline.yml` :

> 💡 **Rappel des concepts vus en cours :**  
> - `runs-on: ubuntu-latest` → GitHub provisionne une VM Linux éphémère (Serverless !)
> - Les `steps` sont **séquentiels** : si l'étape Silver échoue, Gold ne tourne jamais
> - `google-github-actions/auth` → authentification GCP sans mot de passe en clair
> - `bq query < fichier.sql` → exécution d'une requête SQL BigQuery depuis le terminal


In [ ]:
# ── .github/workflows/medallion_pipeline.yml ─────────────────────────────────
# Ce fichier définit le pipeline de données orchestré par GitHub Actions.
# À coller dans .github/workflows/medallion_pipeline.yml

name: "Medallion Pipeline (Bronze → Silver → Gold)"

on:
  # Déclenchement manuel depuis l'onglet Actions (pour le lab)
  workflow_dispatch:

  # Déclenchement automatique chaque nuit à 02:00 UTC (pour la production)
  schedule:
    - cron: '0 2 * * *'

jobs:
  run_medallion_pipeline:
    name: "Transformer les données BigQuery"
    runs-on: ubuntu-latest       # VM Linux éphémère provisionée par GitHub (Serverless !)

    steps:

      # ── Étape 0 : Récupérer le code du dépôt ────────────────────────────────
      - name: "Checkout du code"
        uses: actions/checkout@v4
        # → Clone votre dépôt sur la VM GitHub

      # ── Étape 1 : S'authentifier à Google Cloud ─────────────────────────────
      - name: "Authentification GCP"
        uses: google-github-actions/auth@v2
        with:
          credentials_json: ${{ secrets.GCP_SA_KEY }}
        # → Injecte les credentials depuis le Secret chiffré
        # → La clé n'apparaît JAMAIS en clair dans les logs

      # ── Étape 2 : Vérifier que bq CLI est disponible ───────────────────────
      - name: "Vérifier bq CLI"
        run: |
          bq version
          echo "Projet GCP : ${{ secrets.GCP_PROJECT_ID }}"
        # bq CLI est pré-installé sur ubuntu-latest → pas de pip install !

      # ── Étape 3 : Bronze → Silver ───────────────────────────────────────────
      - name: "[1/3] Bronze → Silver (nettoyage & déduplication)"
        run: |
          echo "Démarrage transformation Bronze → Silver..."
          bq query \
            --project_id="${{ secrets.GCP_PROJECT_ID }}" \
            --location=EU \
            --use_legacy_sql=false \
            --format=pretty \
            < sql/01_bronze_to_silver.sql
          echo "✅ silver.events créée avec succès"
        # Si cette étape échoue → les étapes suivantes sont automatiquement ignorées

      # ── Étape 4 : Silver → Gold ─────────────────────────────────────────────
      - name: "[2/3] Silver → Gold (agrégation KPIs)"
        run: |
          echo "Démarrage agrégation Silver → Gold..."
          bq query \
            --project_id="${{ secrets.GCP_PROJECT_ID }}" \
            --location=EU \
            --use_legacy_sql=false \
            --format=pretty \
            < sql/02_silver_to_gold.sql
          echo "✅ gold.daily_kpis créée avec succès"

      # ── Étape 5 : Validation qualité ────────────────────────────────────────
      - name: "[3/3] Validation qualité (Gold)"
        run: |
          echo "Vérification qualité de gold.daily_kpis..."
          RESULT=$(bq query \
            --project_id="${{ secrets.GCP_PROJECT_ID }}" \
            --location=EU \
            --use_legacy_sql=false \
            --format=json \
            < sql/03_validate_gold.sql)

          echo "Résultat de la validation : $RESULT"

          # Vérifier que la requête a retourné au moins 1 ligne (HAVING passé)
          ROW_COUNT=$(echo "$RESULT" | python3 -c "import sys,json; data=json.load(sys.stdin); print(len(data))")
          if [ "$ROW_COUNT" -eq 0 ]; then
            echo "❌ ÉCHEC : La validation qualité a détecté des anomalies !"
            exit 1   # Force l'échec du pipeline → notification GitHub
          fi
          echo "✅ Validation qualité réussie — $ROW_COUNT règle(s) satisfaite(s)"

      # ── Étape 6 : Résumé final ──────────────────────────────────────────────
      - name: "Résumé du pipeline"
        if: success()    # S'exécute uniquement si toutes les étapes précédentes ont réussi
        run: |
          echo "================================================"
          echo "  Pipeline Medallion terminé avec succès !"
          echo "  ✅ bronze.raw_events  → source des données"
          echo "  ✅ silver.events      → nettoyée + dédupliquée"
          echo "  ✅ gold.daily_kpis    → KPIs prêts pour Looker"
          echo "  Exécuté le : $(date -u '+%Y-%m-%d %H:%M:%S UTC')"
          echo "================================================"


> **🧠 Comprendre le DAG implicite du workflow**
>
> Les `steps` d'un job GitHub Actions sont **séquentiels par défaut**.  
> C'est votre DAG ! Visualisez-le ainsi :
>
> ```
> [Checkout] → [Auth GCP] → [Bronze→Silver] → [Silver→Gold] → [Validation] → [Résumé]
>                                    │                │               │
>                               Si ÉCHEC          Si ÉCHEC       Si ÉCHEC
>                                    ↓                ↓               ↓
>                               Pipeline STOP    Pipeline STOP   Pipeline STOP
>                               + Notification   + Notification  + Notification
>                               GitHub rouge     GitHub rouge    GitHub rouge
> ```
>
> ✅ **Silver ne tourne jamais sur un Bronze vide ou corrompu.**  
> ✅ **Gold ne tourne jamais sur un Silver avec erreur.**


---
## 📤 Partie 4 — Pousser et Déclencher le Pipeline

### 4.1 — Committer et pousser le code


In [ ]:
# Depuis votre terminal local, dans le dossier bigdata-pipeline/

# Vérifier les fichiers créés
ls -la sql/
ls -la .github/workflows/

# Ajouter tous les fichiers
git add .

# Committer
git commit -m "feat: pipeline Medallion orchestré par GitHub Actions

- sql/01_bronze_to_silver.sql : nettoyage, typage, déduplication
- sql/02_silver_to_gold.sql   : agrégation KPIs quotidiens
- sql/03_validate_gold.sql    : validation qualité (HAVING)
- .github/workflows/medallion_pipeline.yml : DAG d'orchestration CI/CD"

# Pousser sur GitHub
git push origin main

echo "✅ Code poussé sur GitHub !"
echo "Rendez-vous sur : https://github.com/<VOTRE_USERNAME>/bigdata-pipeline/actions"


### 4.2 — Déclencher le pipeline manuellement

1. Allez sur votre dépôt GitHub → onglet **Actions**
2. Dans la liste à gauche : **"Medallion Pipeline (Bronze → Silver → Gold)"**
3. Cliquez le bouton bleu **"Run workflow"** → **"Run workflow"**
4. Un point jaune ⬤ apparaît → cliquez dessus pour suivre les logs en direct

**Ce que vous devriez voir dans les logs :**

```
✅ Checkout du code            → 1s
✅ Authentification GCP        → 3s
✅ Vérifier bq CLI             → 2s
✅ [1/3] Bronze → Silver       → 8-15s  ← votre requête SQL tourne dans BigQuery !
✅ [2/3] Silver → Gold         → 5-10s
✅ [3/3] Validation qualité    → 3-5s
✅ Résumé du pipeline          → 1s
─────────────────────────────────────
Total : ~30-40 secondes
```

> 🎯 **C'est ça, le Serverless !**  
> Une VM Linux a été provisionnée automatiquement, votre code SQL a été exécuté sur BigQuery,  
> et la VM a été **détruite**. Vous n'avez rien installé, rien configuré, rien payé de fixe.


---
## 🔬 Partie 5 — Observer et Comprendre

### 5.1 — Vérifier les tables créées dans BigQuery


In [ ]:
-- Vérifier silver.events créée par le pipeline
SELECT
  event_date,
  COUNT(*) AS lignes,
  COUNT(DISTINCT user_id) AS utilisateurs_uniques,
  ROUND(SUM(amount), 2) AS revenu
FROM `silver.events`
GROUP BY 1
ORDER BY 1 DESC;
-- → Si le pipeline a réussi, vous verrez des données


In [ ]:
-- Vérifier gold.daily_kpis — Inspecter le champ refreshed_at
SELECT
  report_date,
  country,
  active_users,
  purchases,
  total_revenue,
  refreshed_at    -- ← horodatage du dernier run GitHub Actions !
FROM `gold.daily_kpis`
ORDER BY report_date DESC, total_revenue DESC;


### 5.2 — Simuler un échec : comprendre la protection du DAG

Modifiez temporairement `sql/03_validate_gold.sql` pour forcer un échec :  
Ajoutez `AND 1 = 0` à la clause `HAVING` pour que la requête ne retourne jamais rien.

```sql
-- Version modifiée pour SIMULER UN ÉCHEC :
HAVING
  COUNT(*) > 0
  AND SUM(total_revenue) > 0
  AND COUNT(CASE WHEN total_revenue < 0 THEN 1 END) = 0
  AND 1 = 0;  -- ← FORCER L'ÉCHEC (pour le test)
```

Puis :
1. Committez et poussez
2. Relancez le pipeline manuellement
3. Observez que l'étape **[3/3] Validation qualité** passe en **rouge ❌**
4. Le pipeline **s'arrête** et GitHub envoie une notification

> **Ce que ça illustre :** en production, si vos données Gold contiennent des anomalies,  
> le pipeline échoue **volontairement** plutôt que de mettre à jour Looker Studio avec des données fausses.

N'oubliez pas de **retirer `AND 1 = 0`** après le test !


### 5.3 — Inspecter les métadonnées du job BigQuery


In [ ]:
export PROJECT_ID=$(gcloud config get-value project)

# Lister les derniers jobs BigQuery — vous verrez ceux déclenchés par GitHub Actions
bq ls --jobs --project_id="${PROJECT_ID}" --max_results=10

# Inspecter un job spécifique (remplacer JOB_ID par l'ID affiché ci-dessus)
# bq show --job=JOB_ID


In [ ]:
-- Vérifier dans INFORMATION_SCHEMA les derniers jobs qui ont touché silver.events
-- (nécessite les droits BigQuery Job Viewer)
SELECT
  job_id,
  creation_time,
  end_time,
  state,
  TIMESTAMP_DIFF(end_time, creation_time, SECOND) AS duree_secondes,
  total_bytes_processed / 1e6 AS mb_scannes,
  destination_table.table_id AS table_cible,
  user_email
FROM `region-EU`.INFORMATION_SCHEMA.JOBS
WHERE
  destination_table.dataset_id IN ('silver', 'gold')
  AND creation_time > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 DAY)
ORDER BY creation_time DESC
LIMIT 10;
-- → Vous verrez les jobs exécutés par le compte de service github-actions-bq@...


---
## ✅ Partie 6 — Checkpoint d'évaluation

### Ce que vous avez construit

```
[GitHub Actions — workflow_dispatch / cron 02:00]
           │
           ▼
   VM Linux éphémère (ubuntu-latest)
   ├── git clone → récupère sql/*.sql
   ├── gcloud auth → credentials GCP sécurisés via Secrets
   │
   ├── [1/3] bq query < sql/01_bronze_to_silver.sql
   │              ↓
   │     silver.events (partitionnée + clusterisée)
   │
   ├── [2/3] bq query < sql/02_silver_to_gold.sql
   │              ↓
   │     gold.daily_kpis (KPIs prêts pour Looker)
   │
   ├── [3/3] bq query < sql/03_validate_gold.sql
   │              ↓
   │     Si 0 ligne → exit 1 → pipeline rouge ❌
   │     Si 1 ligne → pipeline vert ✅
   │
   └── VM détruite automatiquement
```

### Checklist

- [ ] **Secrets GitHub configurés** : `GCP_SA_KEY` et `GCP_PROJECT_ID`
- [ ] **Trois fichiers SQL** créés dans `sql/`
- [ ] **Workflow YAML** créé dans `.github/workflows/`
- [ ] **Premier run réussi** : ✅ vert dans l'onglet Actions
- [ ] **Tables BigQuery vérifiées** : `silver.events` et `gold.daily_kpis` existent
- [ ] **Échec simulé** : le pipeline s'est bien arrêté quand la validation a échoué
- [ ] `AND 1 = 0` **retiré** du fichier de validation

### Tableau de comparaison des approches vues

| Approche | Env. à gérer | Coût fixe | Idéal pour |
|----------|-------------|-----------|------------|
| **Scheduled Query** (J2) | Aucun | Aucun | SQL simple, pas de dépendances |
| **GitHub Actions + bq CLI** (ce lab) | Aucun | Aucun | SQL avec orchestration DAG |
| **GitHub Actions + dbt** | Python, profiles.yml | Aucun | SQL avancé avec tests & lineage |
| **Apache Airflow** | Serveur dédié | ~400€/mois | Pipelines complexes en production |

### Questions de réflexion

1. **Pourquoi n'a-t-on pas eu besoin de `pip install` dans ce pipeline ?**
2. **Quelle est la différence entre ce DAG (steps séquentiels) et un vrai DAG Airflow ?**
3. **Comment ajouteriez-vous une notification Slack si le pipeline échoue ?**
4. **Si `bronze.raw_events` est vide, que se passe-t-il avec `silver.events` ?**
5. **Comment rendre le pipeline incrémental** (ne traiter que les nouvelles données) **plutôt que de tout recalculer** ?
